# Implementation of Multilayer community detection using RID-Rihaczek method

The method we will develop here is from Bullmore, E. & Sporns, O. Complex brain networks: Graph theoretical analysis of structural and functional systems. Nature Rev. Neurosci. 10, 186–198 (2009).

The method goes in two steps: building the multilayer EEG network and then computing the multilayer modularity.

## Intralayer building

For a multilayer brain network where each layer corresponds to a different frequency band, the intra-layer edges correspond to functional connectivity and can be quantified using measures of correlation, coherence or phase synchrony. In prior work, we have illustrated the superior performance of reduced interference Rihaczek (RID-Rihaczek) time-frequency distribution-based phase synchrony index, i.e. RID-TFPS, in terms of time and frequency resolution and robustness to noise. This complex time-frequency distribution can be utilized to calculate the phase difference $\phi_{u, v}(t, f)$, between two signals $x_u$ and $x_v$ as:
$$
\phi_{u, v}(t, f)=\arg \left[\frac{C_u(t, f) C_v^*(t, f)}{\left|C_u(t, f)\right|\left|C_v(t, f)\right|}\right],
$$
where $C_u(t, f)$ and $C_v(t, f)$ are the complex time-frequency distributions of $x_u$ and $x_v$, respectively. Phase Locking Value (PLV) quantifies the consistency of the phase differences across trials and is computed as follows:
$$
\operatorname{PLV}_{u, v}(t, f)=\frac{1}{K}\left|\sum_{k=1}^K e^{j \phi_{u, v}^k(t, f)}\right|
$$
where $K$ is the total number of trials and $\phi_{u, v}^k(t, f)$ is the phase difference between $x_u^k$ and $x_v^k$ for trial $k$. After the pairwise PLV values are computed, the average pairwise synchrony within a predefined time window of interest, $W=\left[t_1, t_2\right]$, and a chosen frequency band is used as intra-layer edge weights, i.c., $w_{u v}^{h h}=\frac{1}{|W|} \frac{1}{|h|} \sum_{t \in W} \sum_{f \in h} \operatorname{PLV}_{u, v}(t, f), 1 \leq u, v \leq N$, where $N$ is the number of brain regions, $|W|$ is the length of the time interval and $|h|$ is the bandwidth of the particular frequency band $h$.

### An FFT-approach

In [1]:
import numpy as np

# Define the number of electrodes
num_electrodes = 16

# Define the duration of the EEG recording (in seconds)
duration = 10

# Define the sampling frequency of the EEG recording (in Hz)
sampling_freq = 1000

# Generate simulated EEG recordings for each electrode
eeg_recordings = {}
for electrode in range(1, num_electrodes + 1):
    # Generate random EEG data
    eeg_data = np.random.randn(int(duration * sampling_freq))
    
    # Store the EEG data in the dictionary
    eeg_recordings[f'Electrode {electrode}'] = eeg_data

# Print the dictionary containing the EEG recordings
print(eeg_recordings)


{'Electrode 1': array([-0.83953752, -0.10501364, -0.91621804, ...,  1.07113458,
        1.02640562, -0.00678317]), 'Electrode 2': array([-0.52871346,  0.51922592, -1.16311415, ...,  0.5220711 ,
       -2.25935752,  1.17130709]), 'Electrode 3': array([-1.49698411,  0.37037689, -1.16823903, ...,  0.2795582 ,
        0.9912168 , -0.86313313]), 'Electrode 4': array([ 0.66049177,  1.83604799,  0.97163392, ...,  1.23104092,
        1.28758496, -1.07346411]), 'Electrode 5': array([-0.42041266, -0.77532922,  0.92394373, ...,  2.86602812,
       -0.0317353 , -0.32097758]), 'Electrode 6': array([-1.795428  ,  0.25694165, -0.16290533, ..., -0.63342333,
       -1.06346316, -0.70801922]), 'Electrode 7': array([ 1.70189451,  0.28021671,  0.53867855, ...,  0.40067694,
       -1.03773752,  0.73555228]), 'Electrode 8': array([ 1.01637217,  0.81281544, -0.63592923, ...,  0.22662873,
        0.70034633, -0.44468337]), 'Electrode 9': array([ 0.05326811, -0.04153093,  1.0245174 , ...,  0.93704085,
       -

In [8]:
import numpy as np

# Define the time window of interest (in seconds)
t1 = 2
t2 = 8

# Define the frequency band of interest (in Hz)
f1 = 8
f2 = 12

# Compute the length of the time window
window_length = t2 - t1

# Compute the number of trials
num_trials = len(eeg_recordings['Electrode 1'])

# Initialize the matrix to store the PLV values
plv_matrix = np.zeros((num_electrodes, num_electrodes))

# Compute the PLV values for each pair of electrodes
for i in range(num_electrodes):
    for j in range(num_electrodes):
        if i != j:
            # Get the EEG data for the current pair of electrodes
            eeg_data_i = eeg_recordings[f'Electrode {i+1}']
            eeg_data_j = eeg_recordings[f'Electrode {j+1}']
            
            # Compute the phase difference between the EEG data
            phase_diff = np.angle(np.fft.fft(eeg_data_i) * np.conj(np.fft.fft(eeg_data_j)))
            print(phase_diff.shape)
            
            # Compute the PLV value
            plv_value = np.abs(np.mean(np.exp(1j * phase_diff)))
            print(plv_value)
            # Compute the average PLV value within the time window and frequency band
            avg_plv_value = np.mean(plv_value[(t1 * sampling_freq):(t2 * sampling_freq), (f1 * sampling_freq):(f2 * sampling_freq)])
            
            # Store the average PLV value in the PLV matrix
            plv_matrix[i, j] = avg_plv_value

# Print the PLV matrix
print(plv_matrix)


(10000,)
0.004725419742081203


IndexError: invalid index to scalar variable.

### A Hilbert approach 

In [ ]:
import numpy as np
from scipy.signal import butter, lfilter, hilbert

# Define the number of electrodes
num_electrodes = 16

# Define the duration of the EEG recording (in seconds)
duration = 10

# Define the sampling frequency of the EEG recording (in Hz)
sampling_freq = 1000

# Generate simulated EEG recordings for each electrode
eeg_recordings = {}
for electrode in range(1, num_electrodes + 1):
    # Generate random EEG data
    eeg_data = np.random.randn(int(duration * sampling_freq))
    # Store the EEG data in the dictionary
    eeg_recordings[f'Electrode {electrode}'] = eeg_data

In [ ]:
# Function to band-pass filter the signal
def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    y = lfilter(b, a, data)
    return y

# Define a frequency band (example: Alpha band)
freq_band = (8, 12)

# Filter data for each electrode
filtered_data = {}
for electrode, data in eeg_recordings.items():
    filtered_data[electrode] = bandpass_filter(data, freq_band[0], freq_band[1], sampling_freq)

In [ ]:
# Compute the Hilbert transform to obtain the analytical signal
analytical_signals = {electrode: hilbert(data) for electrode, data in filtered_data.items()}

# Calculate Phase Differences and PLV
def compute_plv(signal1, signal2):
    phase_diff = np.angle(signal1) - np.angle(signal2)
    plv = np.abs(np.sum(np.exp(1j * phase_diff))) / len(phase_diff)
    return plv

plvs = {}
for i in range(1, num_electrodes + 1):
    for j in range(i + 1, num_electrodes + 1):
        plv = compute_plv(analytical_signals[f'Electrode {i}'], analytical_signals[f'Electrode {j}'])
        plvs[(i, j)] = plv

print(plvs)

## Interlayer building